In [ ]:
# 패키지 설치

%pip install -q langchain langchain-openai python-dotenv
# %pip --> 노트북 환경에 맞게 노트북 안에서 파이썬 패키지 설치 프로그램을 실행

In [ ]:
# 실행환경 감지

try:
    import google.colab     # Colab에만 있는 도구 상자 불러오기 
    IS_COLAB = True         # 성공했다면 Colab이라는 메모를 변수에 저장
    print("Google Colab")
except ImportError:         # 불러오기 실패가 났을때만 실행
    IS_COLAB = False        # Colab이 아니라는 메모를 저장
    print("로컬 Jupyter")

In [ ]:
# API 키 불러오기

import os     # 운영체제와 소통하는 도구 불러옴, 환경변수

if IS_COLAB:  # 앞에서 저장한 메모가 Colab이면 아래 실행
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get["OPENAI_API_KEY"]
    #OPENAI_API_KEY를 꺼내 환경변수 메모장에 같은 이름으로 기록
else: # Colab이 아니라면 아래 실행
    from dotenv import load_dotenv
    # .env 파일 읽기 도구 가져옴
    load_dotenv() #OPENAI_API_KEY를 읽어 환경변수로 등록

print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

In [ ]:
# 필요한 도구 불러오기

from langchain_openai import ChatOpenAI
#langchain_openai 상자에서 ChatOpenAI라는 도구 하나만 꺼냄
from langchain_core.prompts import ChatPromptTemplate
# 프롬프트 템플릿(양식)을 만드는 도구 꺼냄
print("import 완료")

In [ ]:
# 핵심 패턴 작동 확인

llm = ChatOpenAI(model="gpt-4o-mini", temperature = 0.3)
# 부품 1. 모델 설정, 창의성 설정

prompt = ChatPromptTemplate.from_messages([
    # 부품 2. 프롬프트 양식, 메시지 목록으로 양식을 만듬
    ("system", "당신은 {role}입니다"),("human", "{question}")
])
# 배역지정 메시지
# 사람의 질문 / user = human

chain = prompt | llm
# 부품 3. 체인 / |(파이프)는 왼쪽 결과를 오른쪽으로 넘겨라 라는 뜻
# 양식에 빈칸을 채우고 ai에게 넘기는 라인 완성

result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가?를 한문장으로"})
# invoke = 실행, 중괄호{}안에 빈칸의 이름과 채울 값을 짝지어 넘김
print(result.content)

In [ ]:
# 페르소나 만들기

PERSONAS = {  # PERSONAS라는 이름의 딕셔너리 만들기 / 대문자 이름 = 변동없는 설정값
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}
print(f"{len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")
# list(PERSONAS.KEYS())--> 키들(분석가, 마케터...)을 뽑아 리스트로 만들어 출력

In [ ]:
# 페르소나 선택

question = "좋은 데이터란 무엇인가?"
#질문 문장을 question 변수에 담음

for name in ["분석가", "개발자"]:
    #리스트 항목을 하나씩 name에 넣어가며 아래쓴 부분 두 번(분석가, 개발자) 실행
    result = chain.invoke({"role": PERSONAS[name],"question":question})
    #PERSONAS 딕셔너리에서 역할 찾아 부여
    print(f"\n[{name}]")  # 줄바꿈, [name]으로 누구의 답변인지 제목으로 설정
    print(result.content) # 답변 본문 출력
    print("-"*60)  # 구분선 표시

In [ ]:
# ask() 함수

def ask(persona_name: str, question: str) -> str:
    #함수가 받는 재료 2개
    # str은 글자여야 한다는 안내표시 / '->str'은 결과를 글자로 돌려준다는 안내
    system_message = PERSONAS[persona_name]
    # 딕셔너리에서 페르소나 이름으로 system 메시지 조회
    result = chain.invoke({"role": system_message, "question":question})
    #역할 지시문을 {role}에, 질문을 {question}에 넣어 체인 실행
    return result.content

In [ ]:
# 테스트
answer = ask("분석가","A/B 테스트를 한줄로 설명")
print(answer)

In [ ]:
# 명령어 처리 handle_command()

def handle_command(user_input: str, current_persona: str):
    # 사용자가 입력한 글자와 현재 페르소나 이름을 받는 함수 정의
    # 결과 = (페르소나 이름, 종료여부)
    parts = user_input.split(maxsplit = 1)
    # 공백 기준으로 문자열을 잘라 리스트로 만듬
    # maxsplit = 1 --> 처음 한번만 자른다
    cmd = parts[0]
    # 조각 중 맨앞을 꺼내 명령어로 저장

    if cmd in ("/quit", "/exit"):
        # 명령이 둘 중 하나면 종료처리
        print("종료")
        return current_persona, True
        # 페르소나는 그대로 유지, 종료 신호를 True로 돌려줌
    
    if cmd == "/persona":
        print(f"현재 페르소나: {current_persona}")
        #일치하면 현재 페르소나 출력
    
    elif cmd == "help": # 위가 아니라면 이걸 보자 라는 조건, 도움말 출력
        print("명령어: /quit, /exit, /change <이름>, /persona, /help")
    
    elif cmd == "/change": #페르소나 변경명령
        new_name = part[1] if len(parts) > 1 else ""
        #조각이 2개 이상이면 바꿀 이름을, 아니면 빈글자 출력
        # /change만 입력한 경우 에러가 나지 않게 함
        if new_name in PERSONAS:
            #딕셔너리의 key 목록에 이름이 있는지 검사
            print(f"페르소나가 '{new_name}'(으)로 변경되었습니다")
            return new_name, False
            # 있으면 페르소나를 새 이름으로 바꾸고, 종료는 하지 않는다
        else:
            print(f"'{new_name}'은 등록된 페르소나가 아닙니다")
            print(f"사용가능: {','.jon(PERSONAS.keys())}")
            #없으면 거절 메시지와 사용가능한 목록을 보여줌
    else:
        print(f"알 수 없는 명령어 : {cmd} (/help 입력)")
        #어떤 명령어도 해당하지 않으면 "알 수 없는 명령어"라고 알림
    return current_persona, False
    # /persona, /help, 실패한 /change 등 변경이 없는 경우의 출구 

In [ ]:
# 대화 루프, 시연 검증

def run_chatbot(initial_persona: str = "분석가"):
    # 챗봇 실행 함수 / = "분석가"는 기본값
    
    persona = initial_persona
    # 페르소나가 누구인지 적는 변수 / 대화 중 변경 가능 
    
    print(f"페르소나 챗봇입니다. (현재:{persona})")
    # \n은 줄바꿈, 페르소나 출력

    print("  /help로 명령어 보기, /quit로 종료\n")
    # 사용법 출력

    while True:  # 참인 동안 계속 반복
        user_input = input("질문>").strip()
        # input은 키보드 입력을 받아주는 함수, "질문>"이라는 안내문구 출력
        # .strip()은 앞뒤 불필요한 공백 제거

        if not user_input:
            #입력없이 엔터만 친 빈 입력이면 빈 글자를 거짓 취급, 아래 실행
            continue # 나머지를 건너뛰고 루프 맨위로 올라가 입력 대기

        if user_input.startswith("/"):
            # startswith--> "~로 시작하는가". /로 시작하면 명령어로 인식

            persona, should_quit = handle_command(user_input, persona)
            # 명령처리 함수가 돌려주는 두 값을 두 변수에 나눠 받음(새 페르소나, 종료 여부)
            
            if should_quit:
                break       # 종료 신호가 True면 break로 무한루프 탈출, 함수 끝
        else: # 명령어가 아니면 일반 질문
            answer = ask(persona, user_input) # 현재 페르소나로 AI에게 질문해 답을 받음
            print(f"[{persona}] {answer}\n")  # "[persona] 답변..." 형태로 출력 후 한줄 비움

In [ ]:
# 도전과제 1. 대화로그 저장

from datetime import datetime
# 현재 시각을 알려주는 날짜, 시간 도구를 불러옴

def save_log(log, path="chat_log.txt"):
    # 로그 리스트를 파일로 저장하는 함수, 기본값 = chat_log.txt
    with open(path, "w", encoding = "utf-8") as f:
        # 파일을 열어 f라고 부름
        # w는 쓰기 모드(기존 내용 덮어씀)
        # encoding = utf-8 --->한글이 깨지지 않게 설정
        # with는 블록이 끝나면 파일을 자동으로 닫음
        f.write("\n".join(log))
        #"\n".join(log) --> 리스트의 항목들을 줄바꿈으로 이어 하나의 글자로 만듬
        # f.write --> 파일에 씀
    print(f"대화 로그 저장: {path} ({len(log)})")
    #저장된 대화 건수를 출력

def run_chatbot(initial_persona = "분석가"):
    persona = initial_persona
    log = [] # 대화를 담을 빈 리스트 준비
    try: # try ~ finally = try안의 내용이 정상종료되든 중간에 에러,중단이 나든 finally 아래는 반드시 실행
        # /quit을 눌러도, 중간에 중단해도 로그가 날아가지 않게 하는 장치
        while True:
            user_input = input("질문>").strip()
            if not user_input:
                continue
            if user_input.startwith("/"):
                persona, should_quit = handle_command(user_input, persona)
                if should_quit:
                    break
            else:
                answer = ask(persona, user_input)
                print(f"[{persona}] {answer}\n")
                log.append(f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {user_input} → {answer}")
                # 질문에 답할때마다 한줄을 리스트 끝에 추가
                # dateteme.now()는 현재 시각, .strftime('%H:%M:%S')는 시:분:초 모양으로 변환
    finally :
        save_log(log)
            

In [ ]:
# 페르소나를 JSON 파일로 분리

{
  "분석가": {
    "system": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "temperature": 0,
    "desc": "수치와 근거 기반 답변"
  },
  "디자이너": {
    "system": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
    "temperature": 0.8,
    "desc": "UX와 시각적 비유로 쉽은 답변"
  }
}

In [ ]:
import json
# JSON(사전 형태의 텍스트 형식)을 다루는 도구 불러옴

def load_personas(path = "personas.json"):
    # 파일 이름을 받는 함수(기본값 = personas.json)
    with open(path, encoding = "utf-8") as f:
        # 파일을 읽기 모드로 열어 f라 부름
        # 한글 인코딩 지정
        return json.load(f)
        # 파일 내용을 파이썬 딕셔너리로 바꿔 돌려줌 (저장 시 json.dump)

PERSONAS = load_personas()
# 시작 시 한번 읽어 PERSONAS를 채움

In [ ]:
# 페르소나별 temperature
# 목표: 분석가는 temperature 0(정확·일관), 디자이너는 0.8(창의적)처럼 페르소나마다 다른 창의성 설정을 씁니다. 
# 페르소나 사전의 형태를 {이름: {"system": ..., "temperature": ...}}로 확장합니다.

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"), ("human", "{question}")])
# 페르소나 문장 자체가 {role} 빈칸에 삽입
# system 부분을 {role} 하나로 둠
# 당신은 {role} 입니다 라는 문장이 중복되어 어색해지는 문제 해결

CHAINS = {}
# 페르소나별 체인을 담을 빈 딕셔너리
for name, info in PERSONAS.items():
    llm = ChatOpenAI(model = "gpt-4o-mini", temperature = info["temperature"], max_tokens = 200)
    # 페르소나마다 창의성 가진 AI 생성
    # info["temperature"]는 페르소나의 설정값(분석가 0, 디자이너 0.8)을 꺼낸 것
    CHAINS[name] = prompt | llm
    # AI와 프롬프트를 이은 체인을 만들어 저장
    # 한번만 만들어 재사용, AI객체를 만드는 비용 없음

def ask(persona_name: str, question: str) -> str:
    # str은 글자여야 한다는 안내표시 / '->str'은 결과를 글자로 돌려준다는 안내
    result = CHAINS[persona_name].invoke({"role": PERSONAS[persona_name]["system"],})
    # 현재 페르소나에 맞는 체인을 꺼내 실행
    # 이중 대괄호--> 외부 딕셔너리에서 이름으로 찾고, 안에서 다시 system 항목을 찾음
    return result.content
    # 답변 글자만 돌려줌

In [ ]:
# /list 명령어

elif cmd == "/list":
# 기존 /persona, /help 사이에 붙이는 새 조건
# 사용자가 /list 입력 시 아래 실행
    print("사용 가능한 페르소나")
    # 제목 한 줄 출력
    for name, info in PERSONAS.items():
        # 페르소나 하나씩 반복
        print(f"-{name}: {info['desc']}")
        # 분석가: 수치와 근거 기반 답변 형태로 출력
        # ''사용--> f스트링에서 ""를 사용해서
        # desc --> personas.json 한줄 소개

In [ ]:
# max_tokens로 답변 길이 제어

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = info["temperature"], max_tokens=200)
# max_tokens = 200 ----> 답변이 200토큰 도달 시 중간에 잘라서 끝냄